In [ ]:
# ============================================================
# CELL 1 — CHECK ENVIRONMENT
# ============================================================

import sys
import os
import platform
import torch
import transformers

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print(f"Python       : {sys.version}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Platform     : {platform.platform()}")

print("\nCUDA")
print(f"CUDA available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version   : {torch.version.cuda}")
    print(
        f"GPU memory     : "
        f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )
else:
    print("WARNING: CUDA/GPU tidak tersedia.")

print("=" * 60)

ENVIRONMENT CHECK
Python       : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch      : 2.11.0+cu128
Transformers : 5.15.1
Platform     : Linux-6.6.122+-x86_64-with-glibc2.35

CUDA
CUDA available : True
GPU            : Tesla T4
CUDA version   : 12.8
GPU memory     : 14.56 GB


In [ ]:
# ============================================================
# CELL 2 — IMPORT LIBRARIES
# ============================================================

import os
import gc
import time
import random
import warnings

import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

In [ ]:
# ============================================================
# CELL 3 — PROJECT PATH


from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08"

DATA_DIR = os.path.join(PROJECT_DIR, "dataset")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
LOG_DIR = os.path.join(PROJECT_DIR, "logs")

print("=" * 60)
print("PROJECT PATH")
print("=" * 60)

print(f"Project directory : {PROJECT_DIR}")
print(f"Dataset directory : {DATA_DIR}")
print(f"Result directory  : {RESULT_DIR}")
print(f"Log directory     : {LOG_DIR}")

print("\nFolder existing akan digunakan.")

Mounted at /content/drive
PROJECT PATH
Project directory : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08
Dataset directory : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset
Result directory  : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/results
Log directory     : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/logs

Folder existing akan digunakan.


In [ ]:
# ============================================================
# CELL 4 — LOAD FIXED DATA SPLITS
# ============================================================

TRAIN_PATH = os.path.join(
    DATA_DIR,
    "splits",
    "train_split.csv"
)

VAL_PATH = os.path.join(
    DATA_DIR,
    "splits",
    "validation_split.csv"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "splits",
    "test_split.csv"
)

print("=" * 60)
print("LOADING FIXED DATA SPLITS")
print("=" * 60)

print(f"Train      : {TRAIN_PATH}")
print(f"Validation : {VAL_PATH}")
print(f"Test       : {TEST_PATH}")

# ============================================================
# LOAD CSV
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("\nJumlah data:")
print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")

print(
    f"Total      : "
    f"{len(train_df) + len(val_df) + len(test_df):,}"
)

print("\nKolom train:")
print(train_df.columns.tolist())

LOADING FIXED DATA SPLITS
Train      : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/splits/train_split.csv
Validation : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/splits/validation_split.csv
Test       : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/dataset/splits/test_split.csv

Jumlah data:
Train      : 3,052
Validation : 381
Test       : 382
Total      : 3,815

Kolom train:
['Komentar_preprocessed', 'Kode Label', 'label']


In [ ]:
# ============================================================
# CELL 5 — LABEL MAPPING & FIXED SPLIT CHECK
# ============================================================

SEED = 42

# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# ============================================================
# LABEL MAPPING
# ============================================================

label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {
    0: "C0",
    1: "C1",
    2: "C2",
    3: "C3",
    4: "C4"
}

# ============================================================
# PASTIKAN LABEL SUDAH NUMERIK
# ============================================================

# Jika CSV fixed split sudah memiliki kolom "label",
# gunakan langsung tanpa melakukan split ulang.

if "label" not in train_df.columns:
    raise ValueError(
        "Kolom 'label' tidak ditemukan pada train_split.csv."
    )

if "label" not in val_df.columns:
    raise ValueError(
        "Kolom 'label' tidak ditemukan pada validation_split.csv."
    )

if "label" not in test_df.columns:
    raise ValueError(
        "Kolom 'label' tidak ditemukan pada test_split.csv."
    )

# ============================================================
# CEK NILAI LABEL
# ============================================================

expected_labels = {0, 1, 2, 3, 4}

for name, split_df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:

    actual_labels = set(
        split_df["label"].dropna().astype(int).unique()
    )

    if actual_labels != expected_labels:
        raise ValueError(
            f"Label pada {name} tidak sesuai. "
            f"Ditemukan: {sorted(actual_labels)}"
        )

# ============================================================
# FINAL CHECK
# ============================================================

print("=" * 60)
print("FIXED SPLIT & LABEL CHECK")
print("=" * 60)

print(f"Train      : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Test       : {len(test_df):,}")
print(
    f"Total      : "
    f"{len(train_df) + len(val_df) + len(test_df):,}"
)

print("\nLabel mapping:")
for label, idx in label2id.items():
    print(f"{label} -> {idx}")

print("\nDistribusi label TRAIN:")
print(train_df["label"].value_counts().sort_index())

print("\nDistribusi label VALIDATION:")
print(val_df["label"].value_counts().sort_index())

print("\nDistribusi label TEST:")
print(test_df["label"].value_counts().sort_index())

print("\nFixed split digunakan tanpa melakukan split ulang.")

FIXED SPLIT & LABEL CHECK
Train      : 3,052
Validation : 381
Test       : 382
Total      : 3,815

Label mapping:
C0 -> 0
C1 -> 1
C2 -> 2
C3 -> 3
C4 -> 4

Distribusi label TRAIN:
label
0    610
1    611
2    610
3    610
4    611
Name: count, dtype: int64

Distribusi label VALIDATION:
label
0    76
1    76
2    77
3    76
4    76
Name: count, dtype: int64

Distribusi label TEST:
label
0    77
1    76
2    76
3    77
4    76
Name: count, dtype: int64

Fixed split digunakan tanpa melakukan split ulang.


In [ ]:
TEXT_COL = "Komentar_preprocessed"

print(f"Text column : {TEXT_COL}")

Text column : Komentar_preprocessed


In [ ]:
# ============================================================
# CELL 13 — CREATE HUGGING FACE DATASETS
# ============================================================

from datasets import Dataset

print("=" * 60)
print("CREATING HUGGING FACE DATASETS")
print("=" * 60)

# Gunakan hanya kolom yang dibutuhkan model
train_hf = Dataset.from_pandas(
    train_df[[TEXT_COL, "label"]],
    preserve_index=False
)

val_hf = Dataset.from_pandas(
    val_df[[TEXT_COL, "label"]],
    preserve_index=False
)

test_hf = Dataset.from_pandas(
    test_df[[TEXT_COL, "label"]],
    preserve_index=False
)

print(f"Train      : {len(train_hf):,}")
print(f"Validation : {len(val_hf):,}")
print(f"Test       : {len(test_hf):,}")

print("\nContoh train:")
print(train_hf[0])

print("\nDataset Hugging Face berhasil dibuat.")

CREATING HUGGING FACE DATASETS
Train      : 3,052
Validation : 381
Test       : 382

Contoh train:
{'Komentar_preprocessed': 'wah sudah parah kesal ini roman2 nya bang fery sampai gemeteran cuy bicara nya', 'label': 0}

Dataset Hugging Face berhasil dibuat.


In [ ]:
# ============================================================
# CELL 7 — MODEL CONFIGURATION
# INDoBERT LARGE P1 & P2
# ============================================================

MODEL_CONFIGS = {
    "IndoBERT Large P1": {
        "model_name": "indobenchmark/indobert-large-p1",
        "batch_size": 4,
    },
    "IndoBERT Large P2": {
        "model_name": "indobenchmark/indobert-large-p2",
        "batch_size": 4,
    },
}

MAX_LENGTH_CANDIDATES = [128, 256, 512]

NUM_LABELS = 5

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print("\nMODEL CONFIGURATION")
print("-" * 60)

for model_label, config in MODEL_CONFIGS.items():
    print(f"{model_label}")
    print(f"  Checkpoint : {config['model_name']}")
    print(f"  Batch size : {config['batch_size']}")
    print()

print("MAX_LENGTH CANDIDATES")
print("-" * 60)
print(MAX_LENGTH_CANDIDATES)

print("\nTotal experiments :",
      len(MODEL_CONFIGS) * len(MAX_LENGTH_CANDIDATES))

EXPERIMENT CONFIGURATION

MODEL CONFIGURATION
------------------------------------------------------------
IndoBERT Large P1
  Checkpoint : indobenchmark/indobert-large-p1
  Batch size : 4

IndoBERT Large P2
  Checkpoint : indobenchmark/indobert-large-p2
  Batch size : 4

MAX_LENGTH CANDIDATES
------------------------------------------------------------
[128, 256, 512]

Total experiments : 6


In [ ]:
# ============================================================
# CELL 8 — TOKENIZATION FUNCTION
# ============================================================

def tokenize_datasets(
    train_dataset,
    validation_dataset,
    test_dataset,
    tokenizer,
    max_length
):
    """
    Tokenisasi train, validation, dan test menggunakan
    tokenizer dari checkpoint model yang sedang diuji.

    Parameters
    ----------
    train_dataset : Hugging Face Dataset
    validation_dataset : Hugging Face Dataset
    test_dataset : Hugging Face Dataset
    tokenizer : AutoTokenizer
        Tokenizer yang sesuai dengan checkpoint model.
    max_length : int
        Kandidat maximum sequence length.

    Returns
    -------
    tokenized_train : Hugging Face Dataset
    tokenized_validation : Hugging Face Dataset
    tokenized_test : Hugging Face Dataset
    """

    def tokenize_function(examples):
        return tokenizer(
            examples[TEXT_COL],
            truncation=True,
            max_length=max_length,
            padding=False
        )

    tokenized_train = train_dataset.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing train | max_length={max_length}"
    )

    tokenized_validation = validation_dataset.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing validation | max_length={max_length}"
    )

    tokenized_test = test_dataset.map(
        tokenize_function,
        batched=True,
        desc=f"Tokenizing test | max_length={max_length}"
    )

    # Kolom teks tidak lagi diperlukan setelah tokenisasi.
    tokenized_train = tokenized_train.remove_columns(
        [TEXT_COL]
    )

    tokenized_validation = tokenized_validation.remove_columns(
        [TEXT_COL]
    )

    tokenized_test = tokenized_test.remove_columns(
        [TEXT_COL]
    )

    return (
        tokenized_train,
        tokenized_validation,
        tokenized_test
    )


print("=" * 60)
print("TOKENIZATION FUNCTION READY")
print("=" * 60)
print("Function : tokenize_datasets()")
print("MAX_LENGTH candidates :", MAX_LENGTH_CANDIDATES)
print("Status : READY")

TOKENIZATION FUNCTION READY
Function : tokenize_datasets()
MAX_LENGTH candidates : [128, 256, 512]
Status : READY


In [ ]:
# ============================================================
# CELL 9 — DATA COLLATOR
# ============================================================

from transformers import DataCollatorWithPadding

def create_data_collator(tokenizer):
    """
    Membuat data collator dengan dynamic padding
    berdasarkan tokenizer model yang sedang digunakan.
    """

    return DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True
    )


print("=" * 60)
print("DATA COLLATOR FUNCTION READY")
print("=" * 60)
print("Collator : DataCollatorWithPadding")
print("Padding  : Dynamic padding per batch")
print("Status   : READY")

DATA COLLATOR FUNCTION READY
Collator : DataCollatorWithPadding
Padding  : Dynamic padding per batch
Status   : READY


In [ ]:
# ============================================================
# CELL 10 — EVALUATION METRICS
# ============================================================

def compute_metrics(eval_pred):
    """
    Menghitung metrik evaluasi untuk klasifikasi C0-C4.

    Macro-F1 digunakan sebagai metrik utama karena
    setiap kelas perlu diperlakukan secara setara.
    """

    logits, labels = eval_pred

    # Prediksi kelas berdasarkan nilai logit tertinggi
    predictions = np.argmax(logits, axis=-1)

    # Precision, Recall, F1 per kelas lalu dirata-ratakan secara macro
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_macro": f1
    }


print("=" * 60)
print("EVALUATION METRICS READY")
print("=" * 60)

print("Metrics:")
print("- Accuracy")
print("- Macro Precision")
print("- Macro Recall")
print("- Macro F1")

print("\nPrimary metric : Macro-F1")
print("Status         : READY")

EVALUATION METRICS READY
Metrics:
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1

Primary metric : Macro-F1
Status         : READY


In [ ]:
# ============================================================
# CELL 18 — TRAINING ARGUMENTS
# EARLY STOPPING
# ============================================================

from transformers import EarlyStoppingCallback


def create_training_arguments(
    experiment_name,
    batch_size,
    output_dir="/content/indobert_tmp"
):
    """
    Membuat TrainingArguments untuk satu eksperimen.

    num_train_epochs digunakan sebagai batas maksimum training,
    sedangkan penghentian aktual ditentukan oleh Early Stopping.
    """

    experiment_output_dir = os.path.join(
        output_dir,
        experiment_name.replace(" ", "_")
    )

    training_args = TrainingArguments(
        output_dir=experiment_output_dir,

        # ====================================================
        # TRAINING
        # ====================================================

        # Batas maksimum epoch.
        # Training dapat berhenti lebih awal melalui
        # EarlyStoppingCallback.
        num_train_epochs=10,

        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,

        learning_rate=2e-5,
        weight_decay=0.01,

        # ====================================================
        # EVALUATION
        # ====================================================

        eval_strategy="epoch",

        # ====================================================
        # BEST MODEL
        # ====================================================

        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        # ====================================================
        # CHECKPOINT
        # ====================================================

        # Hanya disimpan sementara di storage lokal Colab.
        save_strategy="epoch",

        # ====================================================
        # LOGGING
        # ====================================================

        logging_strategy="epoch",

        # ====================================================
        # GPU / EFFICIENCY
        # ====================================================

        fp16=torch.cuda.is_available(),

        # ====================================================
        # REPRODUCIBILITY
        # ====================================================

        seed=SEED,

        # ====================================================
        # REPORTING
        # ====================================================

        report_to="none",

        # ====================================================
        # DATALOADER
        # ====================================================

        dataloader_pin_memory=torch.cuda.is_available(),
    )

    return training_args


# ============================================================
# EARLY STOPPING CONFIGURATION
# ============================================================

EARLY_STOPPING_PATIENCE = 2

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)


print("=" * 60)
print("TRAINING CONFIGURATION READY")
print("=" * 60)

print("Maximum epochs      : 10")
print("Early stopping      : ENABLED")
print(f"Patience            : {EARLY_STOPPING_PATIENCE}")
print("Evaluation          : every epoch")
print("Best metric         : f1_macro")
print("Learning rate       : 2e-5")
print("Weight decay        : 0.01")
print("FP16                :", torch.cuda.is_available())
print("Checkpoint          : temporary local Colab storage")
print("Google Drive model  : NOT SAVED")
print("Status              : READY")

TRAINING CONFIGURATION READY
Maximum epochs      : 10
Early stopping      : ENABLED
Patience            : 2
Evaluation          : every epoch
Best metric         : f1_macro
Learning rate       : 2e-5
Weight decay        : 0.01
FP16                : True
Checkpoint          : temporary local Colab storage
Google Drive model  : NOT SAVED
Status              : READY


In [ ]:
# ============================================================
# CELL 19 — PREPARE ONE EXPERIMENT
# ============================================================

def prepare_experiment(
    model_label,
    max_length
):
    """
    Menyiapkan satu eksperimen fine-tuning.

    Pipeline:
    tokenizer
    → tokenisasi
    → data collator
    → model
    → TrainingArguments
    → Trainer + Early Stopping

    Training belum dijalankan di fungsi ini.
    """

    # ========================================================
    # GET MODEL CONFIGURATION
    # ========================================================

    if model_label not in MODEL_CONFIGS:
        raise ValueError(
            f"Model tidak ditemukan: {model_label}"
        )

    if max_length not in MAX_LENGTH_CANDIDATES:
        raise ValueError(
            f"MAX_LENGTH tidak valid: {max_length}. "
            f"Gunakan salah satu: {MAX_LENGTH_CANDIDATES}"
        )

    config = MODEL_CONFIGS[model_label]

    model_name = config["model_name"]
    batch_size = config["batch_size"]

    print("=" * 60)
    print("PREPARING EXPERIMENT")
    print("=" * 60)

    print(f"Model      : {model_label}")
    print(f"Checkpoint : {model_name}")
    print(f"MAX_LENGTH : {max_length}")
    print(f"Batch size : {batch_size}")

    # ========================================================
    # 1. LOAD TOKENIZER
    # ========================================================

    print("\n[1/5] Loading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True
    )

    print(
        f"Tokenizer  : {type(tokenizer).__name__}"
    )

    # ========================================================
    # 2. TOKENIZATION
    # ========================================================

    print("\n[2/5] Tokenizing datasets...")

    (
        tokenized_train,
        tokenized_validation,
        tokenized_test
    ) = tokenize_datasets(
        train_hf,
        val_hf,
        test_hf,
        tokenizer,
        max_length
    )

    # ========================================================
    # 3. DATA COLLATOR
    # ========================================================

    print("\n[3/5] Creating data collator...")

    data_collator = create_data_collator(
        tokenizer
    )

    # ========================================================
    # 4. LOAD MODEL
    # ========================================================

    print("\n[4/5] Loading model...")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id
    )

    print(
        f"Model loaded : {model.__class__.__name__}"
    )

    # ========================================================
    # 5. TRAINING ARGUMENTS
    # ========================================================

    print("\n[5/5] Creating TrainingArguments...")

    experiment_name = (
        f"{model_label.replace(' ', '_')}_MAX{max_length}"
    )

    training_args = create_training_arguments(
        experiment_name=experiment_name,
        batch_size=batch_size
    )

    # ========================================================
    # NEW EARLY STOPPING CALLBACK
    # ========================================================

    experiment_early_stopping = EarlyStoppingCallback(
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )

    # ========================================================
    # CREATE TRAINER
    # ========================================================

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_validation,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[experiment_early_stopping]
    )

    print("\n" + "=" * 60)
    print("EXPERIMENT READY")
    print("=" * 60)

    return {
        "model": model,
        "tokenizer": tokenizer,
        "tokenized_train": tokenized_train,
        "tokenized_validation": tokenized_validation,
        "tokenized_test": tokenized_test,
        "data_collator": data_collator,
        "training_args": training_args,
        "trainer": trainer,
        "model_label": model_label,
        "max_length": max_length,
        "experiment_name": experiment_name,
    }

In [ ]:
# ============================================================
# CELL 20 — RUN ONE EXPERIMENT
# ============================================================

def run_experiment(
    model_label,
    max_length
):
    """
    Menjalankan satu eksperimen:
    prepare → train → validation → save metrics → cleanup GPU.
    """

    experiment_name = (
        f"{model_label.replace(' ', '_')}_MAX{max_length}"
    )

    print("\n" + "=" * 70)
    print("START EXPERIMENT")
    print("=" * 70)
    print(f"Experiment : {experiment_name}")
    print(f"Model      : {model_label}")
    print(f"MAX_LENGTH : {max_length}")

    start_time = time.time()

    experiment = None
    trainer = None
    model = None

    try:
        # ====================================================
        # 1. PREPARE
        # ====================================================

        experiment = prepare_experiment(
            model_label=model_label,
            max_length=max_length
        )

        trainer = experiment["trainer"]
        model = experiment["model"]

        # ====================================================
        # 2. TRAIN
        # ====================================================

        print("\n" + "=" * 60)
        print("TRAINING STARTED")
        print("=" * 60)

        train_result = trainer.train()

        # ====================================================
        # 3. VALIDATION
        # ====================================================

        print("\n" + "=" * 60)
        print("VALIDATION")
        print("=" * 60)

        eval_results = trainer.evaluate(
            eval_dataset=experiment["tokenized_validation"]
        )

        # ====================================================
        # 4. BEST CHECKPOINT / EPOCH
        # ====================================================

        best_checkpoint = getattr(
            trainer.state,
            "best_model_checkpoint",
            None
        )

        best_epoch = None

        if best_checkpoint is not None:
            checkpoint_name = os.path.basename(
                best_checkpoint
            )

            if checkpoint_name.startswith("checkpoint-"):
                try:
                    best_step = int(
                        checkpoint_name.split("-")[-1]
                    )

                    best_epoch = (
                        best_step /
                        trainer.state.max_steps
                        * trainer.state.num_train_epochs
                    )

                except Exception:
                    best_epoch = None

        # Jika Trainer mencatat epoch terbaik secara langsung
        if best_epoch is None:
            if hasattr(trainer.state, "best_model_checkpoint"):
                if trainer.state.best_model_checkpoint:
                    # Cari epoch dari log history yang memiliki
                    # metric evaluasi terbaik
                    eval_logs = [
                        log for log in trainer.state.log_history
                        if "eval_f1_macro" in log
                    ]

                    if len(eval_logs) > 0:
                        best_log = max(
                            eval_logs,
                            key=lambda x: x["eval_f1_macro"]
                        )

                        best_epoch = best_log.get("epoch")

        # ====================================================
        # 5. TRAINING TIME
        # ====================================================

        training_time = time.time() - start_time

        # ====================================================
        # 6. EXTRACT METRICS
        # ====================================================

        result_row = {
            "experiment_id": experiment_name,
            "model": model_label,
            "checkpoint": MODEL_CONFIGS[model_label]["model_name"],
            "max_length": max_length,
            "batch_size": MODEL_CONFIGS[model_label]["batch_size"],
            "status": "completed",
            "best_epoch": best_epoch,
            "eval_loss": eval_results.get("eval_loss"),
            "accuracy": eval_results.get("eval_accuracy"),
            "precision": eval_results.get("eval_precision"),
            "recall": eval_results.get("eval_recall"),
            "f1_macro": eval_results.get("eval_f1_macro"),
            "training_time_seconds": training_time,
            "error": ""
        }

        # ====================================================
        # 7. SAVE RESULT IMMEDIATELY
        # ====================================================

        result_df = pd.DataFrame(
            [result_row],
            columns=RESULT_COLUMNS
        )

        result_df.to_csv(
            RESULTS_PATH,
            mode="a",
            header=False,
            index=False
        )

        print("\n" + "=" * 60)
        print("EXPERIMENT COMPLETED")
        print("=" * 60)

        print(f"Model       : {model_label}")
        print(f"MAX_LENGTH  : {max_length}")
        print(f"Best epoch  : {best_epoch}")
        print(
            f"Accuracy    : "
            f"{result_row['accuracy']:.4f}"
        )
        print(
            f"Precision   : "
            f"{result_row['precision']:.4f}"
        )
        print(
            f"Recall      : "
            f"{result_row['recall']:.4f}"
        )
        print(
            f"Macro-F1    : "
            f"{result_row['f1_macro']:.4f}"
        )
        print(
            f"Time        : "
            f"{training_time / 60:.2f} minutes"
        )

        print(f"\nResult saved to:")
        print(RESULTS_PATH)

        return result_row

    except Exception as e:

        training_time = time.time() - start_time

        error_message = repr(e)

        print("\n" + "=" * 60)
        print("EXPERIMENT FAILED")
        print("=" * 60)

        print(f"Model      : {model_label}")
        print(f"MAX_LENGTH : {max_length}")
        print(f"Error      : {error_message}")

        # Simpan status FAILED supaya eksperimen
        # tidak dianggap belum pernah dijalankan.
        failed_row = {
            "experiment_id": experiment_name,
            "model": model_label,
            "checkpoint": MODEL_CONFIGS[model_label]["model_name"],
            "max_length": max_length,
            "batch_size": MODEL_CONFIGS[model_label]["batch_size"],
            "status": "failed",
            "best_epoch": None,
            "eval_loss": None,
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1_macro": None,
            "training_time_seconds": training_time,
            "error": error_message
        }

        pd.DataFrame(
            [failed_row],
            columns=RESULT_COLUMNS
        ).to_csv(
            RESULTS_PATH,
            mode="a",
            header=False,
            index=False
        )

        return failed_row

    finally:

        # ====================================================
        # 8. GPU / MEMORY CLEANUP
        # ====================================================

        print("\nCleaning experiment resources...")

        if trainer is not None:
            del trainer

        if model is not None:
            del model

        if experiment is not None:
            del experiment

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

        print("GPU cleanup completed.")

In [ ]:
# ============================================================
# CELL 14 — EXPERIMENT RESULT TRACKING
# ============================================================

RESULTS_PATH = os.path.join(
    RESULT_DIR,
    "indobert_18_experiments_results.csv"
)

RESULT_COLUMNS = [
    "experiment_id",
    "model",
    "checkpoint",
    "max_length",
    "batch_size",
    "status",
    "best_epoch",
    "eval_loss",
    "accuracy",
    "precision",
    "recall",
    "f1_macro",
    "training_time_seconds",
    "error"
]

# ============================================================
# LOAD / CREATE RESULTS FILE
# ============================================================

if not os.path.exists(RESULTS_PATH):

    results_df = pd.DataFrame(
        columns=RESULT_COLUMNS
    )

    results_df.to_csv(
        RESULTS_PATH,
        index=False
    )

    print("Result file baru dibuat.")

else:

    results_df = pd.read_csv(
        RESULTS_PATH
    )

    print("Result file sudah ada dan akan digunakan.")

# ============================================================
# CHECK
# ============================================================

print("=" * 60)
print("EXPERIMENT RESULT TRACKING")
print("=" * 60)

print(f"Results path : {RESULTS_PATH}")
print(
    f"Recorded experiments : "
    f"{len(results_df)}"
)

if len(results_df) > 0:
    display(results_df)
else:
    print("Belum ada hasil eksperimen.")

Result file sudah ada dan akan digunakan.
EXPERIMENT RESULT TRACKING
Results path : /content/drive/MyDrive/TA EKUIVALENSI/Revisi 27 08/results/indobert_18_experiments_results.csv
Recorded experiments : 15


,experiment_id,model,checkpoint,max_length,batch_size,status,best_epoch,eval_loss,accuracy,precision,recall,f1_macro,training_time_seconds,error
0,IndoBERT_Lite_P1_MAX128,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,128,16,completed,5.0,1.537792,0.270341,0.322164,0.270711,0.245238,91.421959,NaN
1,IndoBERT_Lite_P1_MAX256,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,256,16,completed,1.0,1.578397,0.244094,0.275853,0.244532,0.220760,46.701091,NaN
2,IndoBERT_Lite_P1_MAX512,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,512,16,completed,3.0,1.542539,0.309711,0.265669,0.309159,0.259564,82.544849,NaN
3,IndoBERT_Lite_P2_MAX128,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,128,16,completed,5.0,1.538006,0.296588,0.237580,0.296890,0.258244,93.236050,NaN
4,IndoBERT_Lite_P2_MAX256,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,256,16,completed,6.0,1.551229,0.296588,0.329775,0.297129,0.259322,119.480417,NaN
5,IndoBERT_Lite_P2_MAX512,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,512,16,completed,2.0,1.553601,0.265092,0.259675,0.265789,0.226610,67.416425,NaN
6,IndoBERT_Base_P1_MAX128,IndoBERT Base P1,indobenchmark/indobert-base-p1,128,16,completed,3.0,1.378493,0.493438,0.501060,0.493609,0.491131,403.371578,NaN
7,IndoBERT_Base_P1_MAX256,IndoBERT Base P1,indobenchmark/indobert-base-p1,256,16,completed,3.0,1.340257,0.485564,0.483709,0.485885,0.480689,331.333117,NaN
8,IndoBERT_Base_P1_MAX512,IndoBERT Base P1,indobenchmark/indobert-base-p1,512,16,completed,3.0,1.335024,0.501312,0.498173,0.501606,0.498900,335.865187,NaN
9,IndoBERT_Base_P2_MAX128,IndoBERT Base P2,indobenchmark/indobert-base-p2,128,16,completed,3.0,1.372166,0.485564,0.485261,0.485783,0.485163,297.657983,NaN


In [ ]:
# ============================================================
# CELL 14.5 — RESET LARGE EXPERIMENT RESULTS
# ============================================================

print("=" * 70)
print("RESET INDOBERT LARGE RESULTS")
print("=" * 70)

# Load hasil yang sekarang
results_df = pd.read_csv(RESULTS_PATH)

print(f"Total sebelum reset : {len(results_df)}")

# ============================================================
# IDENTIFIKASI EKSPERIMEN LARGE
# ============================================================

large_mask = results_df["model"].isin([
    "IndoBERT Large P1",
    "IndoBERT Large P2"
])

large_results = results_df[large_mask]

print(f"Hasil Large yang akan dihapus : {len(large_results)}")

if len(large_results) > 0:
    print("\nEksperimen Large yang di-reset:")
    display(
        large_results[
            [
                "experiment_id",
                "model",
                "max_length",
                "status"
            ]
        ]
    )

# ============================================================
# HAPUS SEMUA HASIL LARGE
# ============================================================

results_df = results_df[
    ~large_mask
].copy()

# ============================================================
# SIMPAN KEMBALI
# ============================================================

results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("\n" + "=" * 70)
print("RESET SELESAI")
print("=" * 70)

print(f"Total setelah reset : {len(results_df)}")

print("\nHasil yang tersisa:")
display(results_df)

RESET INDOBERT LARGE RESULTS
Total sebelum reset : 18
Hasil Large yang akan dihapus : 6

Eksperimen Large yang di-reset:


,experiment_id,model,max_length,status
12,IndoBERT_Large_P1_MAX128,IndoBERT Large P1,128,completed
13,IndoBERT_Large_P1_MAX256,IndoBERT Large P1,256,failed
14,IndoBERT_Large_P1_MAX512,IndoBERT Large P1,512,failed
15,IndoBERT_Large_P2_MAX128,IndoBERT Large P2,128,failed
16,IndoBERT_Large_P2_MAX256,IndoBERT Large P2,256,failed
17,IndoBERT_Large_P2_MAX512,IndoBERT Large P2,512,failed



RESET SELESAI
Total setelah reset : 12

Hasil yang tersisa:


,experiment_id,model,checkpoint,max_length,batch_size,status,best_epoch,eval_loss,accuracy,precision,recall,f1_macro,training_time_seconds,error
0,IndoBERT_Lite_P1_MAX128,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,128,16,completed,5.0,1.537792,0.270341,0.322164,0.270711,0.245238,91.421959,NaN
1,IndoBERT_Lite_P1_MAX256,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,256,16,completed,1.0,1.578397,0.244094,0.275853,0.244532,0.220760,46.701091,NaN
2,IndoBERT_Lite_P1_MAX512,IndoBERT Lite P1,indobenchmark/indobert-lite-base-p1,512,16,completed,3.0,1.542539,0.309711,0.265669,0.309159,0.259564,82.544849,NaN
3,IndoBERT_Lite_P2_MAX128,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,128,16,completed,5.0,1.538006,0.296588,0.237580,0.296890,0.258244,93.236050,NaN
4,IndoBERT_Lite_P2_MAX256,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,256,16,completed,6.0,1.551229,0.296588,0.329775,0.297129,0.259322,119.480417,NaN
5,IndoBERT_Lite_P2_MAX512,IndoBERT Lite P2,indobenchmark/indobert-lite-base-p2,512,16,completed,2.0,1.553601,0.265092,0.259675,0.265789,0.226610,67.416425,NaN
6,IndoBERT_Base_P1_MAX128,IndoBERT Base P1,indobenchmark/indobert-base-p1,128,16,completed,3.0,1.378493,0.493438,0.501060,0.493609,0.491131,403.371578,NaN
7,IndoBERT_Base_P1_MAX256,IndoBERT Base P1,indobenchmark/indobert-base-p1,256,16,completed,3.0,1.340257,0.485564,0.483709,0.485885,0.480689,331.333117,NaN
8,IndoBERT_Base_P1_MAX512,IndoBERT Base P1,indobenchmark/indobert-base-p1,512,16,completed,3.0,1.335024,0.501312,0.498173,0.501606,0.498900,335.865187,NaN
9,IndoBERT_Base_P2_MAX128,IndoBERT Base P2,indobenchmark/indobert-base-p2,128,16,completed,3.0,1.372166,0.485564,0.485261,0.485783,0.485163,297.657983,NaN


In [ ]:
# ============================================================
# CELL 15 — RUN 6 FINAL EXPERIMENTS
# 2 MODELS × 3 MAX_LENGTH
# ============================================================

TOTAL_EXPERIMENTS = (
    len(MODEL_CONFIGS) * len(MAX_LENGTH_CANDIDATES)
)

print("=" * 70)
print("FINAL EXPERIMENTS — INDOBERT LARGE")
print("=" * 70)

print("Models     :", list(MODEL_CONFIGS.keys()))
print("MAX_LENGTH :", MAX_LENGTH_CANDIDATES)
print("Total      :", TOTAL_EXPERIMENTS)

# ============================================================
# LOAD EXISTING RESULTS
# ============================================================

results_df = pd.read_csv(RESULTS_PATH)

completed_experiments = set(
    results_df.loc[
        results_df["status"] == "completed",
        "experiment_id"
    ].astype(str)
)

print(
    f"\nCompleted experiments already recorded : "
    f"{len(completed_experiments)}"
)

if len(completed_experiments) > 0:

    print("\nExperiments yang akan di-skip:")

    for exp in sorted(completed_experiments):
        print(f"- {exp}")

# ============================================================
# BUILD EXPERIMENT LIST
# ============================================================

experiment_list = []

experiment_number = 1

for model_label in MODEL_CONFIGS:

    for max_length in MAX_LENGTH_CANDIDATES:

        experiment_name = (
            f"{model_label.replace(' ', '_')}_MAX{max_length}"
        )

        experiment_list.append({
            "number": experiment_number,
            "experiment_id": experiment_name,
            "model_label": model_label,
            "max_length": max_length
        })

        experiment_number += 1

# ============================================================
# RUN EXPERIMENTS
# ============================================================

for exp in experiment_list:

    experiment_number = exp["number"]
    experiment_id = exp["experiment_id"]
    model_label = exp["model_label"]
    max_length = exp["max_length"]

    # --------------------------------------------------------
    # SKIP COMPLETED EXPERIMENT
    # --------------------------------------------------------

    if experiment_id in completed_experiments:

        print("\n" + "=" * 70)
        print(
            f"SKIP EXPERIMENT "
            f"{experiment_number}/{TOTAL_EXPERIMENTS}"
        )
        print("=" * 70)

        print(
            f"{experiment_id} already completed."
        )

        continue

    # --------------------------------------------------------
    # RUN EXPERIMENT
    # --------------------------------------------------------

    print("\n\n" + "#" * 70)
    print(
        f"EXPERIMENT "
        f"{experiment_number}/{TOTAL_EXPERIMENTS}"
    )
    print("#" * 70)

    print(f"Model      : {model_label}")
    print(f"MAX_LENGTH : {max_length}")

    result = run_experiment(
        model_label=model_label,
        max_length=max_length
    )

    # --------------------------------------------------------
    # UPDATE COMPLETED SET
    # --------------------------------------------------------

    if result["status"] == "completed":

        completed_experiments.add(
            experiment_id
        )

    # --------------------------------------------------------
    # SHOW PROGRESS
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print(
        f"PROGRESS: "
        f"{len(completed_experiments)}/"
        f"{TOTAL_EXPERIMENTS} completed"
    )
    print("-" * 70)


# ============================================================
# FINAL SUMMARY
# ============================================================

results_df = pd.read_csv(
    RESULTS_PATH
)

print("\n" + "=" * 70)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 70)

print(
    f"Total recorded : "
    f"{len(results_df)}"
)

print(
    f"Completed      : "
    f"{(results_df['status'] == 'completed').sum()}"
)

print(
    f"Failed         : "
    f"{(results_df['status'] == 'failed').sum()}"
)

display(results_df)

FINAL EXPERIMENTS — INDOBERT LARGE
Models     : ['IndoBERT Large P1', 'IndoBERT Large P2']
MAX_LENGTH : [128, 256, 512]
Total      : 6

Completed experiments already recorded : 12

Experiments yang akan di-skip:
- IndoBERT_Base_P1_MAX128
- IndoBERT_Base_P1_MAX256
- IndoBERT_Base_P1_MAX512
- IndoBERT_Base_P2_MAX128
- IndoBERT_Base_P2_MAX256
- IndoBERT_Base_P2_MAX512
- IndoBERT_Lite_P1_MAX128
- IndoBERT_Lite_P1_MAX256
- IndoBERT_Lite_P1_MAX512
- IndoBERT_Lite_P2_MAX128
- IndoBERT_Lite_P2_MAX256
- IndoBERT_Lite_P2_MAX512


######################################################################
EXPERIMENT 1/6
######################################################################
Model      : IndoBERT Large P1
MAX_LENGTH : 128

START EXPERIMENT
Experiment : IndoBERT_Large_P1_MAX128
Model      : IndoBERT Large P1
MAX_LENGTH : 128
PREPARING EXPERIMENT
Model      : IndoBERT Large P1
Checkpoint : indobenchmark/indobert-large-p1
MAX_LENGTH : 128
Batch size : 4

[1/5] Loading tokenizer...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,1.413173,1.362227,0.419948,0.416065,0.420984,0.380402
2,1.016130,1.303035,0.485564,0.475112,0.486329,0.471161


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


EXPERIMENT FAILED
Model      : IndoBERT Large P2
MAX_LENGTH : 128
Error      : SafetensorError('Error while serializing: I/O error: No space left on device (os error 28)')

Cleaning experiment resources...
GPU cleanup completed.


OSError: [Errno 28] No space left on device